In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from transformers import TextStreamer
from tqdm.auto import tqdm

In [2]:
model_name = "google/gemma-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=True)

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16, #to make memory efficient
    trust_remote_code = False #to ensure security when loading code from the model repository 
)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [6]:
# trying annotations
prompt = ('''
I am an excellent linguist. The task is to label
location entities in the given sentence.
Below are some examples.
Input:Columbus is a city
Output:@@Columbus## is a city
Input:Rare Hendrix song sells for $17
Output:
''')

inputs = tokenizer(prompt, return_tensors="pt")

generated_ids = model.generate(
    inputs["input_ids"],
    max_new_tokens=5000,
    do_sample=True,
)

decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Write to file with annotation
with open("synthetic/annotations.txt", "a", encoding="utf-8") as f:
    f.write(decoded)

print(decoded)


Input:Columbus is a city
Output:@@Columbus## is a city
Input:Rare Hendrix song sells for $17
Output:
Rare Hendrix song sells for **$17**
Input:The world is round
Output:
The world is round **as it is**
Input:The answer is 42
Output:
The answer is 42 **is**
The context is important in natural language processing (NLP) because it determines how a language model understands and responds to a given input.


In [67]:
from faker import Faker

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

print(f"Agreement Date: {agreement_date}")
print(f"Lender: {lender}, Address: {lender_address}")
print(f"Borrower: {borrower}, Address: {borrower_address}")

Agreement Date: July 07, 2025
Lender: Kim, Perry and Smith, Address: 19368 Craig Alley, Staceyside, ND 82505
Borrower: Fisher and Sons, Address: 0097 Archer Course Suite 202, Daletown, GU 70217


In [17]:
from faker import Faker
from transformers import AutoTokenizer, AutoModelForCausalLM

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

topic = (
    f"Loan Agreement\n"
    f"Lender: {lender}, {lender_address}\n"
    f"Borrower: {borrower}, {borrower_address}\n"
    f"Agreement Date: {agreement_date}\n"
)

sections = [
    "0. INTRODUCTION",
    "1. PARTIES",
    "2. DEFINITIONS",
    "3. LENDING DISCLOSURE",
    "4. LOAN TERMS",
    "5. REPAYMENT TERMS",
    "6. INTEREST RATES AND FEES",
    "7. COLLATERAL",
    "8. COVENANTS",
    "9. DEFAULT AND REMEDIES",
    "10. MISCELLANEOUS PROVISIONS"
]

output_file = "synthetic/generated_contract.txt"
contract_so_far = ""  # This accumulates the contract's text as context.

with open(output_file, "w", encoding="utf-8") as f:
    f.write("=== SYNTHETIC CONTRACT GENERATION ===\n\n")
    f.write(f"{topic}\n")
    f.write("="*40 + "\n\n")

for section in sections:
    # Build the prompt using prior generated contract content for context (trim if very long!)
    # For small models, keep context window in mind (e.g. use "contract_so_far[-1500:]" for long contracts)
    prompt = (
        "You are a legal document drafting assistant.\n"
        "You need to generate a full, realistic financial agreement contract section by section. Each section should be coherent with the previous sections and follow legal language and style.\n"
        f"Content:{topic}\nContract so far: {contract_so_far[-1500:]}\n"  # Only use last ~1500 chars for context to avoid overflow.
        f"Generate only the text for the following section: {section}\n"
        # "Continue the contract with this section, following legal language and style.\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        inputs["input_ids"], max_new_tokens=400, do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Remove prompt echo (if present): get only the new text after the prompt
    if decoded.startswith(prompt):
        section_text = decoded[len(prompt):].strip()
    else:
        section_text = decoded.strip()

    # For the next iteration, accumulate the contract so far
    contract_so_far += f"\n\n{section}\n{section_text}"

    # Write to file with annotation
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(f"\n\n--- Prompt for {section} ---\n")
        f.write(prompt.strip() + "\n")
        f.write(f"--- Output for {section} ---\n")
        f.write(section_text)
        f.write("\n" + "="*40 + "\n")

print(f"Done! See the generated contract in: {output_file}")

Done! See the generated contract in: synthetic/generated_contract.txt
